# 步驟三：統計方法與假說檢定

本階段我們將探討：「高嚴重度的氣候事件，是否會造成顯著較高的經濟損失？」
$$ H_o : 高嚴重度的氣候是事件與經濟損失沒有影響$$
$$ H_1 : 高嚴重度的氣候是事件與造成較高的經濟損失$$

### 1. 檢定方法：獨立雙樣本 t 檢定 (Welch's t-test)
考量到高低嚴重度事件的變異數可能不相等，我們採用不假設變異數同質性的 Welch's t-test。

### 2. 效應量分析：Cohen's d。
除了 p-value，我們進一步計算 Cohen's d 來衡量兩組平均數差異的實際影響大小。
公式為： 
$$d = \frac{\bar{x}_1 - \bar{x}_2}{s_p}$$
其中 $s_p$ 為合併標準差 (Pooled Standard Deviation)。

In [18]:
import pandas as pd
from scipy import stats
from pathlib import Path
import sys
import os


# 檢查目前是不是在 notebooks 資料夾裡，如果是，就退回/workspace
if os.getcwd().endswith('notebooks'):
    os.chdir('..')
# 確保能載入 src
BASE_DIR = Path.cwd() 
sys.path.append(str(BASE_DIR))

# 匯入剛剛封裝好的自訂統計函數
from src.stats_analysis import calculate_cohens_d

# 1. 讀取資料，這邊不用標準化過的資料，為了了讓經濟損失的單位是百萬美元，方便解讀
file_path = BASE_DIR  / 'data' / 'processed' / 'global_climate_events_economic_impact_2020_2025_processed.csv'
df = pd.read_csv(file_path)

# 2. 定義群體，分成高於平均的災害損失 vs. 低於或等於平均的災害損失
high_severity_loss = df[df['severity'] > df['severity'].mean()]['economic_impact_million_usd']
low_severity_loss = df[df['severity'] <= df['severity'].mean()]['economic_impact_million_usd']

# 3. 執行檢定與效應量計算
t_stat, p_value = stats.ttest_ind(high_severity_loss, low_severity_loss, equal_var=False, alternative='greater')
d_value = calculate_cohens_d(high_severity_loss, low_severity_loss) 

print(f"p-value:   {p_value:.4e}")
print(f"Cohen's d: {d_value:.4f}")

p-value:   3.0158e-08
Cohen's d: 0.1898


### 💡 統計檢定結果解讀
p-value:   3.0158e-08\
Cohen's d: 0.1898

1. **顯著性 (p-value):** 如果 p-value 遠小於 0.05，代表高嚴重度與低嚴重度災損的差異「在統計上是顯著的」，並非隨機抽樣造成的誤差。
2. **實質意義 (Cohen's d)：** - $d \approx 0.2$ (小效應)
   - $d \approx 0.5$ (中效應)
   - $d \geq 0.8$ (大效應)
3. **結論** 我們可以看到P值遠小於0.05，所以我們拒絕虛無假設$H_o$，但Cohen's d < 0.2，代表統計上沒有顯著的效應
4. 另外，我們發現用standarized的資料和做出來的Cohen'd也是一樣的，經過計算後發現是等價的